In [46]:
# Import necessary libraries
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
import warnings
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error


In [47]:
pd.set_option('display.max_columns', None)

In [48]:
import sys
import os

sys.path.append(os.path.abspath(".."))  
from Resources.properly_format_data import GetData

In [49]:
# Combine lists of plate appearance stats and non-plate appearance stats
additional_stats = ['PA', 'Barrels', 'LD', 'GB', 'FB']
stats_to_measure = ['H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'HBP', 'SF', 'SH', 'SB', 'AB', 'IBB', 'SO']
labels = ['IDfg', 'Season', 'Name', 'Team', 'Age']
combined_list = stats_to_measure + ['Barrels', 'LD', 'GB', 'FB']

In [50]:
data_set = GetData(2021, 2025, additional_stats, stats_to_measure, labels)

In [51]:
formatted_df = data_set.format_data_for_models(add_2026=True)

In [52]:
formatted_df = formatted_df.fillna(0)

In [53]:
#Use Marcel method to create a per pa stat weighted and regressed to the mean

for stat in combined_list:
    player_weighted_stat = ((5*(formatted_df[f'1Prev_{stat}'])) + (4*(formatted_df[f'2Prev_{stat}'])) + (3*(formatted_df[f'3Prev_{stat}'])))/12

    player_weighted_pa = ((formatted_df['1Prev_PA']*5) + (formatted_df['2Prev_PA']*4) + (formatted_df['3Prev_PA']*3))
    league_rate = (
        (formatted_df[f'Prev_1yr_League_Totals_{stat}'] / formatted_df['Prev_1yr_League_Totals_PA']) * formatted_df['1Prev_PA'] * 5 +
        (formatted_df[f'Prev_2yr_League_Totals_{stat}'] / formatted_df['Prev_2yr_League_Totals_PA']) * formatted_df['2Prev_PA'] * 4 +
        (formatted_df[f'Prev_3yr_League_Totals_{stat}'] / formatted_df['Prev_3yr_League_Totals_PA']) * formatted_df['3Prev_PA'] * 3
    ) / player_weighted_pa

    regressed_rate = (player_weighted_stat + (league_rate*100))/((player_weighted_pa/12)+100)    

    formatted_df[f'regressed_rate_{stat}'] = regressed_rate

In [54]:
regressed_cols = [col for col in list(formatted_df.columns) if 'regressed_rate_' in col]

In [55]:
formatted_df = formatted_df[labels + ['PA'] + combined_list + regressed_cols].copy()

In [56]:
for col in combined_list:
    formatted_df[col] = formatted_df[col]/formatted_df['PA']

In [57]:
formatted_df = formatted_df.dropna()

In [58]:
df_to_be_used = formatted_df

## Automating

In [59]:
formatted_df_no_2025 = df_to_be_used[df_to_be_used['Season'] != 2025]
features_2025_df = df_to_be_used[df_to_be_used['Season'] == 2025]

In [60]:
projection_df = formatted_df_no_2025[labels].copy()
projection_2025_df = features_2025_df[labels+['PA']].copy()
X = formatted_df_no_2025[regressed_cols]
X_2025 = features_2025_df[regressed_cols]

for stat in stats_to_measure:

    y = formatted_df_no_2025[stat]  

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    rf_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    projections_full = rf_model.predict(X)
    projection_df[f'{stat}_Proj'] = projections_full

    projections_2025 = rf_model.predict(X_2025)
    projection_2025_df[stat] = projections_2025

    test_pred = rf_model.predict(X_test)
    mae_test = mean_absolute_error(y_test, test_pred)
    print(f"{stat}: Test MAE: {mae_test:.3f}")


H: Test MAE: 0.044
1B: Test MAE: 0.037
2B: Test MAE: 0.015
3B: Test MAE: 0.003
HR: Test MAE: 0.012
R: Test MAE: 0.029
RBI: Test MAE: 0.030
BB: Test MAE: 0.029
HBP: Test MAE: 0.007
SF: Test MAE: 0.005
SH: Test MAE: 0.015
SB: Test MAE: 0.011
AB: Test MAE: 0.038
IBB: Test MAE: 0.002
SO: Test MAE: 0.072


In [61]:
for stat in stats_to_measure:
    projection_2025_df[stat] = projection_2025_df[stat] * projection_2025_df['PA']

In [62]:
projection_2025_df['wOBA'] = ((.691*projection_2025_df['BB']) + (.722*projection_2025_df['HBP']) + (.882*projection_2025_df['1B']) + (1.252*projection_2025_df['2B']) + (1.584*projection_2025_df['3B']) + (2.037*projection_2025_df['HR']))/(projection_2025_df['AB'] + projection_2025_df['BB'] - projection_2025_df['IBB'] + projection_2025_df['SF'] + projection_2025_df['HBP'])

In [63]:
projection_2025_df.head()

,IDfg,Season,Name,Team,Age,PA,H,1B,2B,3B,HR,R,RBI,BB,HBP,SF,SH,SB,AB,IBB,SO,wOBA
44,10155,2025,Mike Trout,LAA,33.0,556.0,118.357579,65.287504,24.567552,2.555259,26.101048,75.836292,71.082376,63.836004,8.287714,3.636533,0.558767,6.492067,476.412101,3.256360,153.648223,0.356433
66,10200,2025,Tucker Barnhart,TEX,34.0,15.0,2.646616,2.103307,0.551322,0.060406,0.168401,1.419751,1.213720,1.343075,0.216257,0.181868,0.368900,0.194805,13.065884,0.017623,4.333791,0.275080
73,10231,2025,Jose Iglesias,SDP,35.0,343.0,90.498384,69.449745,16.199325,1.938116,4.887465,40.468418,31.157540,16.231847,6.678996,1.786429,1.110000,6.229955,319.552004,0.970467,43.215975,0.322188
82,10243,2025,Randal Grichuk,- - -,33.0,293.0,68.971852,42.867307,15.707048,1.033074,10.031647,34.824535,36.491247,18.456083,3.521382,2.059936,0.159819,3.371089,269.264196,0.674522,58.486679,0.324101
112,10324,2025,Marcell Ozuna,ATL,34.0,592.0,140.334611,82.766493,29.160205,1.137285,29.633535,72.861359,87.417060,55.598606,5.274209,4.717260,0.033406,3.688214,527.268877,4.309773,141.191067,0.363436


In [64]:
projection_2025_df.to_csv('../correct wOBA comparison/my_system.csv', index=False)